In [15]:
from collections import defaultdict
from typing import Hashable, Iterable, Sequence, Tuple, Dict, List, Literal

In [16]:
Token = Hashable
Ngram = Tuple[Token, ...]

In [17]:
def _group_ngrams_by_len(common_max_ngrams: Iterable[Sequence[Token]]) -> Dict[int, set[Ngram]]:
    by_len: Dict[int, set[Ngram]] = defaultdict(set)
    for ng in common_max_ngrams:
        t = tuple(ng)
        if len(t) > 0:
            by_len[len(t)].add(t)
    return dict(by_len)


def _merge_intervals(intervals: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    if not intervals:
        return []
    intervals.sort()
    merged = [intervals[0]]
    for s, e in intervals[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged


def _covered_token_count(tokens: Sequence[Token], ngrams_by_len: Dict[int, set[Ngram]]) -> int:
    L = len(tokens)
    if L == 0 or not ngrams_by_len:
        return 0

    intervals: List[Tuple[int, int]] = []

    # For each length, do one sliding pass and record matched spans.
    for n, grams in ngrams_by_len.items():
        if n > L:
            continue
        # Slide windows of size n
        for i in range(0, L - n + 1):
            if tuple(tokens[i : i + n]) in grams:
                intervals.append((i, i + n))

    # Union spans (prevents double-counting overlapping matches)
    merged = _merge_intervals(intervals)
    return sum(e - s for s, e in merged)


def weighted_max_common_score(
    q_tokens: Sequence[Token],
    k_tokens: Sequence[Token],
    common_max_ngrams: Iterable[Sequence[Token]],
    *,
    overall: Literal["harmonic", "mean", "min", "max"] = "harmonic",
    decimals: int = 6,
) -> Dict[str, float]:
    """
    Best-fit score for your setup:
      - uses ONLY maximal common n-grams you already computed
      - weights by n automatically via token coverage
      - symmetric overall score (recommended: harmonic mean)

    Returns: {score, q_cov, k_cov}
    """
    ngrams_by_len = _group_ngrams_by_len(common_max_ngrams)

    if not q_tokens or not k_tokens or not ngrams_by_len:
        return {"score": 0.0, "q_cov": 0.0, "k_cov": 0.0}

    q_cov = _covered_token_count(q_tokens, ngrams_by_len) / len(q_tokens)
    k_cov = _covered_token_count(k_tokens, ngrams_by_len) / len(k_tokens)

    if overall == "harmonic":
        score = 0.0 if (q_cov + k_cov) == 0 else (2 * q_cov * k_cov) / (q_cov + k_cov)
    elif overall == "mean":
        score = (q_cov + k_cov) / 2
    elif overall == "min":
        score = min(q_cov, k_cov)
    elif overall == "max":
        score = max(q_cov, k_cov)
    else:
        raise ValueError("overall must be 'harmonic', 'mean', 'min', or 'max'.")

    return {
        "score": round(score, decimals),
        "q_cov": round(q_cov, decimals),
        "k_cov": round(k_cov, decimals),
    }

In [18]:
import sys

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_txt
from n_gram_tracing import (
    common_ngrams
)

In [19]:
tokenizer = load_model("/Volumes/BCross/models/gpt2", load_model=False)

In [50]:
known_text = read_txt("../../data/hodja_nasreddin_text_1.txt")
unknown_text = read_txt("../../data/hodja_nasreddin_text_10.txt")

In [51]:
common = common_ngrams(
    text1=known_text,
    text2=unknown_text,
    tokenizer=tokenizer,
    include_subgrams=False,
    lowercase=True
)

In [53]:
def get_tokens(
    text: str,
    tokenizer,
    *,
    lowercase: bool = True,
    add_special_tokens: bool = False,
):
    """
    Hugging Face Transformers tokenizer -> list of token strings.

    Uses:
      - tokenizer.encode(..., add_special_tokens=...)  :contentReference[oaicite:0]{index=0}
      - tokenizer.convert_ids_to_tokens(...)            :contentReference[oaicite:1]{index=1}
    """
    if lowercase:
        text = text.lower()

    input_ids = tokenizer.encode(text, add_special_tokens=add_special_tokens)
    return tokenizer.convert_ids_to_tokens(input_ids)

In [54]:
unknown_tokens = get_tokens(unknown_text, tokenizer)
known_tokens = get_tokens(known_text, tokenizer)

In [25]:
from collections import defaultdict
from typing import Hashable, Iterable, Sequence, Tuple, Dict, List, Literal, Any

Token = Hashable
Ngram = Tuple[Token, ...]


def _group_ngrams_by_len(common_max_ngrams: Iterable[Sequence[Token]]) -> Dict[int, set[Ngram]]:
    by_len: Dict[int, set[Ngram]] = defaultdict(set)
    for ng in common_max_ngrams:
        t = tuple(ng)
        if len(t) > 0:
            by_len[len(t)].add(t)
    return dict(by_len)


def _merge_intervals(intervals: List[Tuple[int, int]]) -> List[Tuple[int, int]]:
    if not intervals:
        return []
    intervals.sort()
    merged = [intervals[0]]
    for s, e in intervals[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged


def _covered_token_count(tokens: Sequence[Token], grams: set[Ngram]) -> int:
    """Unique token positions covered by ANY occurrence of any n-gram in `grams`."""
    L = len(tokens)
    if L == 0 or not grams:
        return 0

    # all grams in this call share the same length
    n = len(next(iter(grams)))
    if n > L:
        return 0

    intervals: List[Tuple[int, int]] = []
    for i in range(0, L - n + 1):
        if tuple(tokens[i : i + n]) in grams:
            intervals.append((i, i + n))

    merged = _merge_intervals(intervals)
    return sum(e - s for s, e in merged)


def _combine(q_cov: float, k_cov: float, overall: str) -> float:
    if overall == "harmonic":
        return 0.0 if (q_cov + k_cov) == 0 else (2 * q_cov * k_cov) / (q_cov + k_cov)
    if overall == "mean":
        return (q_cov + k_cov) / 2
    if overall == "min":
        return min(q_cov, k_cov)
    if overall == "max":
        return max(q_cov, k_cov)
    raise ValueError("overall must be 'harmonic', 'mean', 'min', or 'max'.")


def weighted_max_common_score_by_n(
    q_tokens: Sequence[Token],
    k_tokens: Sequence[Token],
    common_max_ngrams: Iterable[Sequence[Token]],
    *,
    overall: Literal["harmonic", "mean", "min", "max"] = "harmonic",
    decimals: int = 6,
) -> Dict[str, Any]:
    """
    Returns:
      - overall: union-coverage score using all common maximal n-grams
      - per_n: list of rows, one per n (length), each with q_cov/k_cov/score

    Weighting: automatic by length because coverage counts token positions.
    """
    ngrams_by_len = _group_ngrams_by_len(common_max_ngrams)

    if not q_tokens or not k_tokens or not ngrams_by_len:
        return {
            "overall": {"score": 0.0, "q_cov": 0.0, "k_cov": 0.0},
            "per_n": [],
        }

    per_n_rows: List[Dict[str, float]] = []
    for n in sorted(ngrams_by_len):
        grams = ngrams_by_len[n]
        q_cov = _covered_token_count(q_tokens, grams) / len(q_tokens)
        k_cov = _covered_token_count(k_tokens, grams) / len(k_tokens)
        score = _combine(q_cov, k_cov, overall)
        per_n_rows.append(
            {
                "n": float(n),
                "q_cov": round(q_cov, decimals),
                "k_cov": round(k_cov, decimals),
                "score": round(score, decimals),
            }
        )

    # overall union coverage across ALL n-grams (don’t double-count overlaps across lengths)
    all_intervals_q: List[Tuple[int, int]] = []
    all_intervals_k: List[Tuple[int, int]] = []

    # build all intervals by scanning per length (still fast enough for typical lengths)
    for n, grams in ngrams_by_len.items():
        if n <= len(q_tokens):
            for i in range(0, len(q_tokens) - n + 1):
                if tuple(q_tokens[i : i + n]) in grams:
                    all_intervals_q.append((i, i + n))
        if n <= len(k_tokens):
            for i in range(0, len(k_tokens) - n + 1):
                if tuple(k_tokens[i : i + n]) in grams:
                    all_intervals_k.append((i, i + n))

    q_cov_all = (sum(e - s for s, e in _merge_intervals(all_intervals_q)) / len(q_tokens)) if q_tokens else 0.0
    k_cov_all = (sum(e - s for s, e in _merge_intervals(all_intervals_k)) / len(k_tokens)) if k_tokens else 0.0
    overall_score = _combine(q_cov_all, k_cov_all, overall)

    return {
        "overall": {
            "score": round(overall_score, decimals),
            "q_cov": round(q_cov_all, decimals),
            "k_cov": round(k_cov_all, decimals),
        },
        "per_n": per_n_rows,
    }

In [55]:
from typing import Hashable, Iterable, Sequence, Tuple, Dict, List
from collections import defaultdict

import pandas as pd

Token = Hashable
Ngram = Tuple[Token, ...]


def _distinct_ngrams(tokens: Sequence[Token], n: int) -> set[Ngram]:
    """Distinct contiguous n-gram *types* (presence/absence)."""
    L = len(tokens)
    if n <= 0 or n > L:
        return set()
    return {tuple(tokens[i : i + n]) for i in range(L - n + 1)}


def ngram_tracing_df_from_common(
    *,
    known_tokens: Sequence[Token],
    unknown_tokens: Sequence[Token],
    common_ngrams: Iterable[Sequence[Token]],
    decimals: int = 3,
    validate_common: bool = True,
) -> pd.DataFrame:
    """
    Per-n output like n-gram tracing (boolean existence), but with overlap supplied by you.

    Output columns:
      token_level,
      known_ngrams_distinct,
      unknown_ngrams_distinct,
      overlap_ngrams_distinct,
      simpson,  (overlap / unknown_distinct)  [idiolect's a/(a+b)]
      jaccard   (overlap / union_distinct)    [idiolect's a/(a+b+c)]
    """
    # group supplied common n-grams by length
    common_by_n: Dict[int, set[Ngram]] = defaultdict(set)
    for ng in common_ngrams:
        t = tuple(ng)
        if len(t) > 0:
            common_by_n[len(t)].add(t)

    rows: List[dict] = []

    for n in sorted(common_by_n):
        K = _distinct_ngrams(known_tokens, n)
        Q = _distinct_ngrams(unknown_tokens, n)

        overlap_set = set(common_by_n[n])

        # Optional safety: ensure passed "common" items actually occur in BOTH
        if validate_common:
            overlap_set &= (K & Q)

        a = len(overlap_set)
        known_cnt = len(K)
        unknown_cnt = len(Q)
        union_cnt = known_cnt + unknown_cnt - a

        simpson = 0.0 if unknown_cnt == 0 else a / unknown_cnt
        jaccard = 0.0 if union_cnt == 0 else a / union_cnt

        rows.append(
            {
                "token_level": n,
                "known_ngrams_distinct": known_cnt,
                "unknown_ngrams_distinct": unknown_cnt,
                "overlap_ngrams_distinct": a,
                "simpson": round(simpson, decimals),
                "jaccard": round(jaccard, decimals),
            }
        )

    return pd.DataFrame(rows)

In [56]:
len(common)

59

In [57]:
df = ngram_tracing_df_from_common(
    unknown_tokens=unknown_tokens,
    known_tokens=known_tokens,
    common_ngrams=common
)

df

,token_level,known_ngrams_distinct,unknown_ngrams_distinct,overlap_ngrams_distinct,simpson,jaccard
0,1,247,238,43,0.181,0.097
1,2,385,388,10,0.026,0.013
2,3,412,430,5,0.012,0.006
3,4,429,447,1,0.002,0.001


In [58]:
sum(df['overlap_ngrams_distinct'])

59

In [59]:
from collections import defaultdict
from typing import Hashable, Iterable, Sequence, Tuple, Dict, List, Literal, Optional

import pandas as pd

Token = Hashable
Ngram = Tuple[Token, ...]


def _distinct_ngrams(tokens: Sequence[Token], n: int) -> set[Ngram]:
    """Distinct contiguous n-gram *types* (presence/absence)."""
    L = len(tokens)
    if n <= 0 or n > L:
        return set()
    return {tuple(tokens[i : i + n]) for i in range(L - n + 1)}


def ngram_tracing_df_from_common_weighted(
    *,
    known_tokens: Sequence[Token],
    unknown_tokens: Sequence[Token],
    common_ngrams: Iterable[Sequence[Token]],
    weight: Literal["linear", "power"] = "linear",
    alpha: float = 1.0,           # used only if weight="power": w(n)=n**alpha
    decimals: int = 3,
    validate_common: bool = True,
) -> pd.DataFrame:
    """
    Per-n table (boolean existence) + weighting components for later aggregation.

    Columns:
      token_level
      known_ngrams_distinct (K_n)
      unknown_ngrams_distinct (Q_n)
      overlap_ngrams_distinct (a_n)
      union_ngrams_distinct (U_n)

      simpson  = a_n / Q_n
      jaccard  = a_n / U_n

      weight_w = w(n)
      num_w = w(n) * a_n
      den_simpson_w = w(n) * Q_n
      den_jaccard_w = w(n) * U_n
    """
    # group supplied common n-grams by length
    common_by_n: Dict[int, set[Ngram]] = defaultdict(set)
    for ng in common_ngrams:
        t = tuple(ng)
        if len(t) > 0:
            common_by_n[len(t)].add(t)

    rows: List[dict] = []

    for n in sorted(common_by_n):
        K = _distinct_ngrams(known_tokens, n)
        Q = _distinct_ngrams(unknown_tokens, n)

        overlap_set = set(common_by_n[n])

        # optional safety: ensure passed items actually occur in BOTH
        if validate_common:
            overlap_set &= (K & Q)

        a = len(overlap_set)
        known_cnt = len(K)
        unknown_cnt = len(Q)
        union_cnt = known_cnt + unknown_cnt - a

        simpson = 0.0 if unknown_cnt == 0 else a / unknown_cnt
        jaccard = 0.0 if union_cnt == 0 else a / union_cnt

        if weight == "linear":
            w = float(n)
        elif weight == "power":
            w = float(n) ** float(alpha)
        else:
            raise ValueError("weight must be 'linear' or 'power'.")

        rows.append(
            {
                "token_level": n,
                "known_ngrams_distinct": known_cnt,
                "unknown_ngrams_distinct": unknown_cnt,
                "overlap_ngrams_distinct": a,
                "union_ngrams_distinct": union_cnt,
                "simpson": round(simpson, decimals),
                "jaccard": round(jaccard, decimals),
                "weight_w": w,
                "num_w": w * a,
                "den_simpson_w": w * unknown_cnt,
                "den_jaccard_w": w * union_cnt,
            }
        )

    return pd.DataFrame(rows)

In [38]:
df_2 = ngram_tracing_df_from_common_weighted(
    unknown_tokens=unknown_tokens,
    known_tokens=known_tokens,
    common_ngrams=common
)

df_2

,token_level,known_ngrams_distinct,unknown_ngrams_distinct,overlap_ngrams_distinct,union_ngrams_distinct,simpson,jaccard,weight_w,num_w,den_simpson_w,den_jaccard_w
0,1,345,420,45,720,0.107,0.062,1.0,45.0,420.0,720.0
1,2,691,902,56,1537,0.062,0.036,2.0,112.0,1804.0,3074.0
2,3,781,1042,17,1806,0.016,0.009,3.0,51.0,3126.0,5418.0
3,4,795,1080,9,1866,0.008,0.005,4.0,36.0,4320.0,7464.0
4,5,800,1090,4,1886,0.004,0.002,5.0,20.0,5450.0,9430.0
5,6,804,1092,2,1894,0.002,0.001,6.0,12.0,6552.0,11364.0
6,7,803,1092,2,1893,0.002,0.001,7.0,14.0,7644.0,13251.0


In [60]:
df_2 = ngram_tracing_df_from_common_weighted(
    unknown_tokens=unknown_tokens,
    known_tokens=known_tokens,
    common_ngrams=common
)

df_2

,token_level,known_ngrams_distinct,unknown_ngrams_distinct,overlap_ngrams_distinct,union_ngrams_distinct,simpson,jaccard,weight_w,num_w,den_simpson_w,den_jaccard_w
0,1,247,238,43,442,0.181,0.097,1.0,43.0,238.0,442.0
1,2,385,388,10,763,0.026,0.013,2.0,20.0,776.0,1526.0
2,3,412,430,5,837,0.012,0.006,3.0,15.0,1290.0,2511.0
3,4,429,447,1,875,0.002,0.001,4.0,4.0,1788.0,3500.0


In [39]:
def aggregate_ge(df: pd.DataFrame, t: int, coef: str = "simpson") -> float:
    sub = df[df["token_level"] >= t]
    if sub.empty:
        return 0.0
    num = sub["num_w"].sum()
    den = sub["den_simpson_w"].sum() if coef == "simpson" else sub["den_jaccard_w"].sum()
    return 0.0 if den == 0 else float(num / den)

In [61]:
results_list = []
levels = df_2['token_level'].drop_duplicates()

for level in levels:
    res_score = aggregate_ge(df_2, level)

    row = {
        "min_token_size": level,
        "score": res_score
    }

    results_list.append(row)

results = pd.DataFrame(results_list)

In [62]:
results

,min_token_size,score
0,1,0.020039
1,2,0.010119
2,3,0.006173
3,4,0.002237


In [49]:
results

,min_token_size,score
0,1,0.009892
1,2,0.008479
2,3,0.004909
3,4,0.003422
4,5,0.002341
5,6,0.001832
6,7,0.001832
